# 第19章：标准 Attention — 理解 O(N^2) 的瓶颈

## 本章概览

在深度学习的 Transformer 架构中，**Self-Attention（自注意力）** 是最核心的计算模块。
然而，标准的 Attention 实现存在严重的内存瓶颈 —— 它需要 **O(N^2)** 的显存来存储注意力矩阵，
其中 N 是序列长度。当 N 增大到数千甚至数万时，这将导致显存溢出（OOM）。

本章我们将：
1. 回顾 Self-Attention 的数学公式
2. 逐步分解计算过程：S -> P -> O
3. 分析内存消耗为何是 O(N^2)
4. 用 Triton 实现一个朴素的 Attention kernel
5. 通过 Benchmark 展示内存随序列长度的二次增长
6. 理解为何这种方法对长序列不可行

这将为后续章节（FlashAttention v1~v4）奠定基础。

## 1. Self-Attention 公式

### 1.1 基本公式

给定查询矩阵 Q、键矩阵 K 和值矩阵 V，Self-Attention 的计算公式为：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

其中：
- **Q** (Query): 形状为 `[N, d]`，N 是序列长度，d 是每个头的维度
- **K** (Key): 形状为 `[N, d]`
- **V** (Value): 形状为 `[N, d]`
- **d_k**: 缩放因子，通常等于 d（头维度），用于防止点积值过大

### 1.2 为什么需要缩放？

当 d 较大时，Q 和 K 的点积值的方差会随 d 增长。
如果不除以 sqrt(d)，softmax 的输入值会很大，
导致梯度接近零（softmax 饱和区），训练困难。

### 1.3 逐步分解

标准 Attention 可以分解为三个步骤：

```
步骤 1:  S = Q @ K^T / sqrt(d)     # 注意力分数矩阵  [N, N]
步骤 2:  P = softmax(S, dim=-1)     # 注意力权重矩阵  [N, N]
步骤 3:  O = P @ V                  # 输出矩阵        [N, d]
```

## 2. 计算流程图（ASCII 图解）

```
    Q [N, d]          K [N, d]           V [N, d]
    |                  |                  |
    |                  | 转置              |
    |                  v                  |
    |              K^T [d, N]             |
    |                  |                  |
    +-------+  +-------+                  |
            |  |                          |
            v  v                          |
         矩阵乘法                         |
     S = Q @ K^T [N, N]                   |
            |                             |
            v                             |
     S = S / sqrt(d)                      |
            |                             |
            v                             |
     (可选) 应用 Causal Mask              |
     S[i,j] = -inf  if j > i             |
            |                             |
            v                             |
     P = softmax(S, dim=-1) [N, N]        |
            |                             |
            +----------+  +---------------+
                       |  |
                       v  v
                    矩阵乘法
                O = P @ V [N, d]
                       |
                       v
                  输出 O [N, d]
```

### 关键观察：S 矩阵是 N x N 的！

这是内存瓶颈的根源。对于序列长度 N=4096，d=128：
- Q, K, V 各需要 N * d = 4096 * 128 = 524,288 个元素
- **S 矩阵需要 N * N = 4096 * 4096 = 16,777,216 个元素** 
- S 矩阵比 Q/K/V 大 32 倍！

当 N=16384 时，S 矩阵需要 16384^2 = 2.68 亿个元素（FP32 约 1GB）。

## 3. 内存分析

### 3.1 各步骤的内存需求

| 张量 | 形状 | 元素数 | FP16 内存 |
|------|------|--------|----------|
| Q | [B, H, N, d] | B*H*N*d | O(N*d) per head |
| K | [B, H, N, d] | B*H*N*d | O(N*d) per head |
| V | [B, H, N, d] | B*H*N*d | O(N*d) per head |
| **S** | **[B, H, N, N]** | **B*H*N*N** | **O(N^2) per head** |
| **P** | **[B, H, N, N]** | **B*H*N*N** | **O(N^2) per head** |
| O | [B, H, N, d] | B*H*N*d | O(N*d) per head |

其中 B=batch size, H=head数, N=序列长度, d=头维度。

### 3.2 内存增长示例（B=1, H=32, d=128, FP16）

```
N=512:    S+P 内存 = 32 * 512^2 * 2 * 2B    =  32 MB     -- OK
N=1024:   S+P 内存 = 32 * 1024^2 * 2 * 2B   = 128 MB     -- OK  
N=2048:   S+P 内存 = 32 * 2048^2 * 2 * 2B   = 512 MB     -- 开始紧张
N=4096:   S+P 内存 = 32 * 4096^2 * 2 * 2B   =   2 GB     -- 很大
N=8192:   S+P 内存 = 32 * 8192^2 * 2 * 2B   =   8 GB     -- 可能 OOM
N=16384:  S+P 内存 = 32 * 16384^2 * 2 * 2B  =  32 GB     -- 必定 OOM
N=65536:  S+P 内存 = 32 * 65536^2 * 2 * 2B  = 512 GB     -- 完全不可能
```

可以看到，**内存以序列长度的平方增长**，这就是 O(N^2) 瓶颈。

In [ ]:
import torch
import triton
import triton.language as tl
import math

print(f"PyTorch version: {torch.__version__}")
print(f"Triton version: {triton.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")

## 4. PyTorch 参考实现

首先，我们用纯 PyTorch 实现标准 Attention，作为正确性验证的参考。

In [ ]:
def attention_pytorch_reference(
    q: torch.Tensor,  # [B, H, N, d]
    k: torch.Tensor,  # [B, H, N, d]
    v: torch.Tensor,  # [B, H, N, d]
    causal: bool = False,
    sm_scale: float = None,
) -> torch.Tensor:
    """
    标准 Attention 的 PyTorch 参考实现。
    
    参数:
        q: Query 张量 [B, H, N, d]
        k: Key 张量   [B, H, N, d]
        v: Value 张量 [B, H, N, d]
        causal: 是否使用因果掩码
        sm_scale: softmax 缩放因子，默认 1/sqrt(d)
    
    返回:
        output: [B, H, N, d]
    """
    B, H, N, d = q.shape
    if sm_scale is None:
        sm_scale = 1.0 / math.sqrt(d)
    
    # 步骤 1: 计算注意力分数 S = Q @ K^T / sqrt(d)
    # q: [B, H, N, d], k^T: [B, H, d, N] => S: [B, H, N, N]
    s = torch.matmul(q, k.transpose(-2, -1)) * sm_scale
    
    # 可选: 应用因果掩码
    if causal:
        # 创建下三角掩码: mask[i,j] = True if j <= i
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    
    # 步骤 2: Softmax
    p = torch.softmax(s, dim=-1)
    
    # 步骤 3: 加权求和 O = P @ V
    # p: [B, H, N, N], v: [B, H, N, d] => o: [B, H, N, d]
    o = torch.matmul(p, v)
    
    return o


# 测试参考实现
torch.manual_seed(42)
B, H, N, d = 2, 4, 128, 64
q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)

out_ref = attention_pytorch_reference(q, k, v, causal=False)
print(f"输入形状: Q={q.shape}, K={k.shape}, V={v.shape}")
print(f"输出形状: {out_ref.shape}")
print(f"输出范围: [{out_ref.min().item():.4f}, {out_ref.max().item():.4f}]")

# 因果注意力测试
out_causal = attention_pytorch_reference(q, k, v, causal=True)
print(f"\n因果注意力输出形状: {out_causal.shape}")
print(f"因果注意力输出范围: [{out_causal.min().item():.4f}, {out_causal.max().item():.4f}]")

## 5. Triton 实现：朴素 Attention

### 5.1 设计思路

我们的朴素 Triton 实现将完整地物化（materialize）N x N 的注意力分数矩阵 S。
每个 Triton program 负责计算输出矩阵 O 的一个行块（BLOCK_M 行）。

```
   程序网格设计:
   +-----------------------------------------+
   |  grid = (cdiv(N, BLOCK_M), B * H)       |
   |                                          |
   |  pid_m = program_id(0)  -> Q 行块索引     |
   |  pid_bh = program_id(1) -> batch*head     |
   +-----------------------------------------+
   
   每个 program 的工作:
   
   Q_block [BLOCK_M, d]       K [N, d]
        |                       |
        +------- matmul --------+
        |                              
        v                              
   S_row [BLOCK_M, N]  <-- 完整的一行！这里就是 O(N) per program
        |                              
        v                              
   P_row = softmax(S_row) [BLOCK_M, N]
        |                              
        +------- matmul --------+
        |                       |
        v                  V [N, d]
   O_block [BLOCK_M, d]
```

注意：每个 program 需要读取完整的 K 和 V 矩阵（N 行），这导致大量全局内存读取。

In [ ]:
@triton.jit
def naive_attention_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    N: tl.constexpr,
    D: tl.constexpr,
    sm_scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    朴素 Attention Triton kernel。
    每个 program 处理 BLOCK_M 行的 Q，遍历所有 K/V 来计算完整的注意力分数行。
    
    注意：此实现会在 SRAM 中物化 BLOCK_M x N 的注意力分数片段，
    通过分块累积的方式避免一次性分配 N x N 的全局内存，
    但仍然是 O(N) per program 的工作量。
    """
    # 获取 program 索引
    pid_m = tl.program_id(0)    # Q 行块索引
    pid_bh = tl.program_id(1)   # batch * head 索引
    
    # 计算 batch 和 head 索引
    # (假设 grid 的 dim1 = B * H)
    
    # 基地址偏移
    q_base = Q_ptr + pid_bh * stride_qh
    k_base = K_ptr + pid_bh * stride_kh
    v_base = V_ptr + pid_bh * stride_vh
    o_base = O_ptr + pid_bh * stride_oh
    
    # Q 块的行范围
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # [BLOCK_M]
    offs_d = tl.arange(0, BLOCK_D)                      # [BLOCK_D]
    
    # 加载 Q 块: [BLOCK_M, BLOCK_D]
    q_ptrs = q_base + offs_m[:, None] * stride_qn + offs_d[None, :] * stride_qd
    q_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
    
    # ============================================
    # 第一遍：计算注意力分数并找到行最大值（用于数值稳定的 softmax）
    # ============================================
    # 初始化行最大值
    row_max = tl.full([BLOCK_M], value=float('-inf'), dtype=tl.float32)
    
    # 遍历所有 K 块，计算 S = Q @ K^T 并找到行最大值
    num_k_blocks = tl.cdiv(N, BLOCK_N)
    
    # 我们需要两遍遍历：
    # 第一遍：找到 row_max 用于稳定 softmax
    # 第二遍：计算 exp(s - max) 并累积
    # 这是朴素实现的特点 —— FlashAttention 通过在线 softmax 只需一遍
    
    for block_n_idx in range(0, num_k_blocks):
        offs_n = block_n_idx * BLOCK_N + tl.arange(0, BLOCK_N)
        
        # 因果掩码：如果当前 K 块的起始位置 > Q 块的最大位置，跳过
        if IS_CAUSAL:
            if block_n_idx * BLOCK_N > (pid_m + 1) * BLOCK_M - 1:
                break
        
        # 加载 K 块: [BLOCK_N, BLOCK_D]
        k_ptrs = k_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        k_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        # 计算 S_block = Q_block @ K_block^T  [BLOCK_M, BLOCK_N]
        s_block = tl.dot(q_block, tl.trans(k_block))
        s_block = s_block * sm_scale
        
        # 因果掩码
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s_block = tl.where(causal_mask, s_block, float('-inf'))
        
        # 有效元素掩码
        valid_mask = offs_n[None, :] < N
        s_block = tl.where(valid_mask, s_block, float('-inf'))
        
        # 更新行最大值
        block_max = tl.max(s_block, axis=1)  # [BLOCK_M]
        row_max = tl.maximum(row_max, block_max)
    
    # ============================================
    # 第二遍：计算 softmax 分母和加权输出
    # ============================================
    row_sum = tl.zeros([BLOCK_M], dtype=tl.float32)    # softmax 分母
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)  # 输出累加器
    
    for block_n_idx in range(0, num_k_blocks):
        offs_n = block_n_idx * BLOCK_N + tl.arange(0, BLOCK_N)
        
        if IS_CAUSAL:
            if block_n_idx * BLOCK_N > (pid_m + 1) * BLOCK_M - 1:
                break
        
        # 重新加载 K 块 (朴素实现的代价：需要两遍读取 K)
        k_ptrs = k_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        k_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        # 重新计算 S_block
        s_block = tl.dot(q_block, tl.trans(k_block))
        s_block = s_block * sm_scale
        
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s_block = tl.where(causal_mask, s_block, float('-inf'))
        
        valid_mask = offs_n[None, :] < N
        s_block = tl.where(valid_mask, s_block, float('-inf'))
        
        # 计算 exp(s - max)
        p_block = tl.exp(s_block - row_max[:, None])  # [BLOCK_M, BLOCK_N]
        
        # 累积分母
        row_sum += tl.sum(p_block, axis=1)  # [BLOCK_M]
        
        # 加载 V 块并累积输出
        v_ptrs = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)
        
        # acc += P_block @ V_block
        p_block_fp16 = p_block.to(tl.float16)
        acc += tl.dot(p_block_fp16, v_block).to(tl.float32)
    
    # 归一化: O = acc / row_sum
    acc = acc / row_sum[:, None]
    
    # 写回输出
    o_ptrs = o_base + offs_m[:, None] * stride_on + offs_d[None, :] * stride_od
    o_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    tl.store(o_ptrs, acc.to(tl.float16), mask=o_mask)

In [ ]:
def naive_attention_triton(
    q: torch.Tensor,  # [B, H, N, d]
    k: torch.Tensor,  # [B, H, N, d]
    v: torch.Tensor,  # [B, H, N, d]
    causal: bool = False,
) -> torch.Tensor:
    """
    朴素 Attention 的 Triton wrapper。
    """
    B, H, N, d = q.shape
    assert k.shape == (B, H, N, d)
    assert v.shape == (B, H, N, d)
    
    # 分配输出
    o = torch.empty_like(q)
    
    sm_scale = 1.0 / math.sqrt(d)
    
    # 块大小配置
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    
    # 网格: (Q 块数, B * H)
    grid = (triton.cdiv(N, BLOCK_M), B * H)
    
    # 将 [B, H, N, d] 视为 [B*H, N, d] 来简化 stride 计算
    # stride_qb 和 stride_qh 合并为一个 stride_qh = N*d
    naive_attention_kernel[grid](
        q, k, v, o,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        o.stride(0), o.stride(1), o.stride(2), o.stride(3),
        N=N,
        D=d,
        sm_scale=sm_scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    
    return o


print("Triton 朴素 Attention kernel 定义完成!")

## 6. 正确性验证

将 Triton 实现与 PyTorch 参考实现进行对比。

In [ ]:
def test_correctness(B, H, N, d, causal=False, dtype=torch.float16):
    """验证 Triton naive attention 与 PyTorch 参考实现的一致性。"""
    torch.manual_seed(42)
    q = torch.randn(B, H, N, d, device='cuda', dtype=dtype)
    k = torch.randn(B, H, N, d, device='cuda', dtype=dtype)
    v = torch.randn(B, H, N, d, device='cuda', dtype=dtype)
    
    # PyTorch 参考实现
    out_ref = attention_pytorch_reference(q, k, v, causal=causal)
    
    # Triton 实现
    out_triton = naive_attention_triton(q, k, v, causal=causal)
    
    # 比较
    max_diff = (out_ref - out_triton).abs().max().item()
    mean_diff = (out_ref - out_triton).abs().mean().item()
    
    # FP16 的容差较大
    rtol = 1e-2 if dtype == torch.float16 else 1e-4
    atol = 1e-2 if dtype == torch.float16 else 1e-4
    is_close = torch.allclose(out_ref, out_triton, rtol=rtol, atol=atol)
    
    causal_str = "因果" if causal else "全量"
    status = "PASS" if is_close else "FAIL"
    print(f"[{status}] B={B}, H={H}, N={N}, d={d}, {causal_str}注意力 | "
          f"最大误差={max_diff:.6f}, 平均误差={mean_diff:.6f}")
    return is_close


# 运行一系列测试
print("=" * 80)
print("正确性测试")
print("=" * 80)

all_pass = True
for N in [64, 128, 256, 512]:
    for causal in [False, True]:
        result = test_correctness(B=2, H=4, N=N, d=64, causal=causal)
        all_pass = all_pass and result

print("\n" + "=" * 80)
if all_pass:
    print("所有测试通过!")
else:
    print("存在测试失败，请检查实现!")
print("=" * 80)

## 7. 内存消耗可视化

让我们实际测量不同序列长度下的 GPU 内存消耗，验证 O(N^2) 的增长趋势。

In [ ]:
def measure_memory_usage(seq_lengths, B=1, H=8, d=64):
    """
    测量标准 Attention（物化完整 S 矩阵）的内存消耗。
    
    注意：这里我们测量的是理论内存（分配 S 和 P 矩阵），
    而不是 Triton kernel 的内存（Triton kernel 使用分块，不分配完整 S 矩阵）。
    """
    results = []
    
    for N in seq_lengths:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        try:
            # 分配输入
            q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
            k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
            v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
            
            # 计算理论内存
            input_mem = 3 * B * H * N * d * 2  # Q, K, V, FP16 = 2 bytes
            s_matrix_mem = B * H * N * N * 2     # S 矩阵, FP16
            p_matrix_mem = B * H * N * N * 2     # P 矩阵, FP16
            output_mem = B * H * N * d * 2       # O 矩阵, FP16
            total_theory = input_mem + s_matrix_mem + p_matrix_mem + output_mem
            
            # 实际执行标准 attention（物化 S 矩阵）
            mem_before = torch.cuda.memory_allocated()
            sm_scale = 1.0 / math.sqrt(d)
            s = torch.matmul(q, k.transpose(-2, -1)) * sm_scale  # [B, H, N, N] 物化！
            p = torch.softmax(s, dim=-1)
            o = torch.matmul(p, v)
            mem_after = torch.cuda.max_memory_allocated()
            actual_mem = mem_after - mem_before
            
            results.append({
                'N': N,
                'input_mem_mb': input_mem / 1e6,
                's_matrix_mem_mb': s_matrix_mem / 1e6,
                'total_theory_mb': total_theory / 1e6,
                'actual_peak_mb': mem_after / 1e6,
                'status': 'OK'
            })
            
            # 释放中间结果
            del s, p, o, q, k, v
            torch.cuda.empty_cache()
            
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                results.append({
                    'N': N,
                    'input_mem_mb': input_mem / 1e6,
                    's_matrix_mem_mb': s_matrix_mem / 1e6,
                    'total_theory_mb': total_theory / 1e6,
                    'actual_peak_mb': float('nan'),
                    'status': 'OOM'
                })
                torch.cuda.empty_cache()
            else:
                raise
    
    return results


# 测量
seq_lengths = [256, 512, 1024, 2048, 4096]
results = measure_memory_usage(seq_lengths, B=1, H=8, d=64)

# 打印结果表格
print("\n" + "=" * 90)
print(f"{'序列长度':>10} | {'输入内存(MB)':>12} | {'S矩阵内存(MB)':>14} | {'理论总内存(MB)':>14} | {'状态':>6}")
print("-" * 90)
for r in results:
    print(f"{r['N']:>10} | {r['input_mem_mb']:>12.2f} | {r['s_matrix_mem_mb']:>14.2f} | "
          f"{r['total_theory_mb']:>14.2f} | {r['status']:>6}")
print("=" * 90)
print("\n注意: S 矩阵内存随 N^2 增长，当 N 翻倍时，S 矩阵内存增加 4 倍！")

## 8. 性能 Benchmark

比较 PyTorch 标准 Attention 和 Triton 朴素 Attention 的性能。

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],
        x_vals=[128, 256, 512, 1024, 2048],
        line_arg='provider',
        line_vals=['pytorch', 'triton_naive'],
        line_names=['PyTorch Standard', 'Triton Naive'],
        styles=[('blue', '-'), ('red', '-')],
        ylabel='ms',
        plot_name='Naive Attention: PyTorch vs Triton',
        args={'B': 2, 'H': 8, 'd': 64},
    )
)
def bench_attention(B, H, N, d, provider):
    q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    
    quantiles = [0.5, 0.2, 0.8]
    
    if provider == 'pytorch':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: attention_pytorch_reference(q, k, v),
            quantiles=quantiles
        )
    elif provider == 'triton_naive':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: naive_attention_triton(q, k, v),
            quantiles=quantiles
        )
    
    return ms, min_ms, max_ms


bench_attention.run(print_data=True)

## 9. 为什么标准 Attention 对长序列不可行？

### 9.1 根本问题：IO 瓶颈

现代 GPU 的算力（FLOPS）远超内存带宽。标准 Attention 的主要问题不仅是内存容量，
还有 **内存读写量**：

```
标准 Attention 的 HBM 读写:

步骤 1: S = Q @ K^T
  - 读取 Q [N, d] 和 K [N, d]    : O(N*d) 读取
  - 写入 S [N, N] 到 HBM          : O(N^2) 写入   <-- 瓶颈!

步骤 2: P = softmax(S)
  - 从 HBM 读取 S [N, N]          : O(N^2) 读取   <-- 瓶颈!
  - 写入 P [N, N] 到 HBM          : O(N^2) 写入   <-- 瓶颈!

步骤 3: O = P @ V
  - 从 HBM 读取 P [N, N]          : O(N^2) 读取   <-- 瓶颈!
  - 读取 V [N, d]                 : O(N*d) 读取
  - 写入 O [N, d]                 : O(N*d) 写入

总 HBM 读写: O(N^2) —— 被 S/P 矩阵主导
```

### 9.2 FlashAttention 的核心思想预览

FlashAttention 的关键洞察：**永远不要在 HBM 中物化 N x N 的注意力矩阵**。

```
标准方法:                    FlashAttention:
Q,K -> S(N,N) -> HBM         Q,K -> S_块 -> SRAM (不写回 HBM!)
S -> P(N,N) -> HBM           S_块 -> P_块 -> SRAM (在线 softmax)
P,V -> O(N,d) -> HBM         P_块,V -> O_块 累加 -> HBM

HBM 读写: O(N^2)             HBM 读写: O(N*d)  
```

FlashAttention 通过分块计算和在线 softmax，将 HBM 读写从 O(N^2) 降低到 O(N*d)，
这是一个巨大的改进！

下一章，我们将详细学习 FlashAttention v1 的实现。

## 10. 练习

### 练习 1: 多头注意力维度理解
给定模型维度 d_model=1024，头数 H=16，计算：
- 每个头的维度 d_k
- 对于序列长度 N=4096，S 矩阵的总大小（包含所有头，FP16）
- Q, K, V 的总大小（FP16）
- S 矩阵是 Q/K/V 的多少倍？

### 练习 2: 因果掩码的计算节省
对于因果注意力，S 矩阵中大约一半的元素被掩码为 -inf。
修改上面的 benchmark，比较有无因果掩码的性能差异。
思考：因果掩码能节省多少计算？

### 练习 3: Softmax 数值稳定性
实现一个不使用 `tl.max` 技巧的 softmax（直接计算 exp(x)/sum(exp(x))），
对比数值稳定版本，观察在大值输入下的结果差异。

### 练习 4: 内存估算
你的 GPU 有 24GB 显存，模型使用 B=1, H=32, d=128。
不考虑其他开销，仅存储 S 和 P 矩阵，最长能支持多长的序列？

提示: 24GB = 24 * 10^9 bytes, FP16 = 2 bytes/element
需要: 2 * 32 * N^2 * 2 <= 24 * 10^9
求解 N。

## 总结

本章我们学习了：

1. **Self-Attention 公式**：Attention(Q,K,V) = softmax(QK^T/sqrt(d)) @ V
2. **三步分解**：S=QK^T (分数) -> P=softmax(S) (权重) -> O=PV (输出)
3. **O(N^2) 瓶颈**：S 和 P 矩阵是 N x N 的，内存和 IO 都随 N^2 增长
4. **朴素 Triton 实现**：通过两遍遍历实现完整的 attention
5. **长序列问题**：N 超过几千时，标准方法面临 OOM

### 核心教训

> **标准 Attention 的瓶颈不在于计算量，而在于中间矩阵 S 的内存占用和 IO 开销。**
> 解决方案：不要物化完整的 S 矩阵 —— 这就是 FlashAttention 的核心思想。

下一章：FlashAttention v1 —— 分块 + 在线 Softmax